# 01 - Exploracion de datos: Amazon Video Games

Objetivo: entender la calidad y estructura del dataset antes de decidir la limpieza para el sistema de recomendacion.

Preguntas guia:

- Que columnas son utiles para el recomendador?
- Cuantos nulos, duplicados o valores fuera de rango hay?
- Como se distribuyen ratings, usuarios, productos y fechas?
- Que reglas de limpieza conservan suficientes datos sin introducir ruido?

## 1. Setup

Este notebook esta pensado para ejecutarse desde `kedro jupyter lab` o directamente desde Jupyter. Si el catalogo de Kedro no esta disponible, carga el parquet crudo desde `data/01_raw/video_games.parquet`.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "01_raw" / "video_games.parquet"

# Algunas comprobaciones exactas requieren shuffles grandes. Por defecto se dejan desactivadas
# para que el notebook sea ejecutable en local con el dataset completo.
RUN_EXPENSIVE_CHECKS = False
SAMPLE_FRACTION = 0.05

PROJECT_ROOT, RAW_DATA_PATH

(WindowsPath('C:/Users/asier/Documents/8ºCD/BIG DATA/amazon-recsys'),
 WindowsPath('C:/Users/asier/Documents/8ºCD/BIG DATA/amazon-recsys/data/01_raw/video_games.parquet'))

In [2]:
spark = (
    SparkSession.builder
    .appName("amazon-recsys-data-exploration")
    .master("local[4]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

In [3]:
try:
    from kedro.framework.session import KedroSession
    from kedro.framework.startup import bootstrap_project

    metadata = bootstrap_project(PROJECT_ROOT)
    with KedroSession.create(project_path=PROJECT_ROOT) as session:
        context = session.load_context()
        catalog = context.catalog
        df = catalog.load("amazon_reviews")
    print("Dataset cargado desde Kedro catalog: amazon_reviews")
except Exception as exc:
    print(f"No se pudo usar Kedro catalog ({type(exc).__name__}: {exc}). Cargando parquet directamente.")
    df = spark.read.parquet(str(RAW_DATA_PATH))

raw_columns = df.columns
analysis_columns = [
    "rating",
    "asin",
    "parent_asin",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase",
]
analysis_columns = [column for column in analysis_columns if column in raw_columns]
df = df.select(*analysis_columns)

# No cacheamos el DataFrame completo: el parquet crudo contiene columnas de texto e imagenes
# que pueden agotar la memoria local. El EDA principal usa columnas ligeras con pushdown.

[04/30/26 11:47:16] INFO     Using 'C:\Users\asier\Documents\8ºCD\BIG                               __init__.py:275
                             DATA\amazon-recsys\.venv\Lib\site-packages\kedro\framework\project\ric                
                             h_logging.yml' as logging configuration.                                              

[04/30/26 11:47:17] INFO     No typed parameter requirements found, returning original   parameter_validator.py:108
                             parameters                                                                            

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving plugin.py:242
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

[04/30/26 11:47:18] INFO     Loading data from amazon_reviews (SparkDataset)...                data_catalog.py:1053

Dataset cargado desde Kedro catalog: amazon_reviews


## 2. Vista general

Primero comprobamos volumen, esquema y unas filas de muestra. Esto ayuda a confirmar si las columnas esperadas para el recomendador existen: `user_id`, `parent_asin` y `rating`.

In [4]:
n_rows = df.count()
n_cols = len(df.columns)

print(f"Filas: {n_rows:,}")
print(f"Columnas: {n_cols:,}")
df.printSchema()

[04/30/26 11:47:24] ERROR    Exception while sending command.                                  java_gateway.py:1055
                             ╭────────────── Traceback (most recent call last) ──────────────╮                     
                             │ C:\Users\asier\Documents\8ºCD\BIG                             │                     
                             │ DATA\amazon-recsys\.venv\Lib\site-packages\py4j\clientserver. │                     
                             │ py:511 in send_command                                        │                     
                             │                                                               │                     
                             │   508 │   │                                                   │                     
                             │   509 │   │   try:                                            │                     
                             │   510 │   │   │   while True:                                 │                     
                             │ ❱ 511 │   │   │   │   answer = smart_decode(self.stream.readl │                     
                             │   512 │   │   │   │   logger.debug("Answer received: {0}".for │                     
                             │   513 │   │   │   │   # Happens when a the other end is dead. │                     
                             │   514 │   │   │   │   # answer before the socket raises an er │                     
                             │                                                               │                     
                             │ C:\Users\asier\AppData\Roaming\uv\python\cpython-3.13.5-windo │                     
                             │ ws-x86_64-none\Lib\socket.py:719 in readinto                  │                     
                             │                                                               │                     
                             │   716 │   │   if self._timeout_occurred:                      │                     
                             │   717 │   │   │   raise OSError("cannot read from timed out o │                     
                             │   718 │   │   try:                                            │                     
                             │ ❱ 719 │   │   │   return self._sock.recv_into(b)              │                     
                             │   720 │   │   except timeout:                                 │                     
                             │   721 │   │   │   self._timeout_occurred = True               │                     
                             │   722 │   │   │   raise                                       │                     
                             ╰───────────────────────────────────────────────────────────────╯                     
                             ConnectionResetError: [WinError 10054] Se ha forzado la                               
                             interrupción de una conexión existente por el host remoto                             
                                                                                                                   
                             During handling of the above exception, another exception                             
                             occurred:                                                                             
                                                                                                                   
                             ╭────────────── Traceback (most recent call last) ──────────────╮                     
                             │ C:\Users\asier\Documents\8ºCD\BIG                             │                     
                             │ DATA\amazon-recsys\.venv\Lib\site-packages\py4j\java_gateway. │                     
                             │ py:1038 in send_command  

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 n_rows = df.count()                                                                          │
│   2 n_cols = len(df.columns)                                                                     │
│   3                                                                                              │
│   4 print(f"Filas: {n_rows:,}")                                                                  │
│                                                                                                  │
│ C:\Users\asier\Documents\8ºCD\BIG                                                                │
│ DATA\amazon-recsys\.venv\Lib\site-packages\pyspark\sql\dataframe.py:1240 in count                │
│                                                                                                  │
│   1237 │   │   >>> df.count()                                                                    │
│   1238 │   │   3                                                                                 │
│   1239 │   │   """                                                                               │
│ ❱ 1240 │   │   return int(self._jdf.count())                                                     │
│   1241 │                                                                                         │
│   1242 │   def collect(self) -> List[Row]:                                                       │
│   1243 │   │   """Returns all the records as a list of :class:`Row`.                             │
│                                                                                                  │
│ C:\Users\asier\Documents\8ºCD\BIG                                                                │
│ DATA\amazon-recsys\.venv\Lib\site-packages\py4j\java_gateway.py:1322 in __call__                 │
│                                                                                                  │
│   1319 │   │   │   proto.END_COMMAND_PART                                                        │
│   1320 │   │                                                                                     │
│   1321 │   │   answer = self.gateway_client.send_command(command)                                │
│ ❱ 1322 │   │   return_value = get_return_value(                                                  │
│   1323 │   │   │   answer, self.gateway_client, self.target_id, self.name)                       │
│   1324 │   │                                                                                     │
│   1325 │   │   for temp_arg in temp_args:                                                        │
│                                                                                                  │
│ C:\Users\asier\Documents\8ºCD\BIG                                                                │
│ DATA\amazon-recsys\.venv\Lib\site-packages\pyspark\errors\exceptions\captured.py:179 in deco     │
│                                                                                                  │
│   176 def capture_sql_exception(f: Callable[..., Any]) -> Callable[..., Any]:                    │
│   177 │   def deco(*a: Any, **kw: Any) -> Any:                                                   │
│   178 │   │   try:                                                                               │
│ ❱ 179 │   │   │   return f(*a, **kw)                                                             │
│   180 │   │   except Py4JJavaError as e:                                                         │
│   181 │   │   │   converted = convert_exception(e.java_exception)                                │
│   182 │   │   │   if not isinstance(converted, UnknownExcep

In [ ]:
df.limit(5).toPandas()

In [ ]:
expected_columns = ["user_id", "asin", "parent_asin", "rating", "timestamp", "verified_purchase", "helpful_vote", "title", "text", "images"]
pd.DataFrame(
    {
        "column": expected_columns,
        "exists_in_raw": [column in raw_columns for column in expected_columns],
        "used_in_main_eda": [column in df.columns for column in expected_columns],
    }
)

## 3. Nulos y cadenas vacias

Para ALS y otros recomendadores colaborativos, los campos minimos son usuario, item y rating. Las columnas de texto pueden ser utiles para analisis posterior, pero no deben bloquear el dataset base si estan vacias.

In [ ]:
missing_exprs = []
for column_name, dtype in df.dtypes:
    is_missing = F.col(column_name).isNull()
    if dtype == "string":
        is_missing = is_missing | (F.trim(F.col(column_name)) == "")
    missing_exprs.append(F.sum(F.when(is_missing, 1).otherwise(0)).alias(column_name))

missing_pd = df.select(missing_exprs).toPandas().T.reset_index()
missing_pd.columns = ["column", "missing_rows"]
missing_pd["missing_pct"] = missing_pd["missing_rows"] / n_rows
missing_pd.sort_values("missing_pct", ascending=False)

In [ ]:
plot_df = missing_pd.sort_values("missing_pct", ascending=True)

plt.figure(figsize=(8, 4))
sns.barplot(data=plot_df, x="missing_pct", y="column", color="#4C78A8")
plt.gca().xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
plt.xlabel("Porcentaje de filas")
plt.ylabel("")
plt.title("Nulos o strings vacios por columna")
plt.tight_layout()

Decision provisional:

- `user_id`, `parent_asin` y `rating` no deberian tener nulos en el dataset modelable.
- `title`, `text` e `images` pueden conservarse para EDA o modelos hibridos, pero no son obligatorias para ALS.
- `asin` identifica variantes concretas; `parent_asin` agrupa producto padre y suele ser mejor item para recomendacion.

## 4. Ratings

Revisamos rango, valores invalidos y distribucion. El pipeline actual castea `rating` a `float`; antes de limpiar conviene comprobar que todos los valores estan en la escala esperada.

In [ ]:
rating_summary = df.select(
    F.count("rating").alias("non_null_ratings"),
    F.min("rating").alias("min_rating"),
    F.max("rating").alias("max_rating"),
    F.mean("rating").alias("mean_rating"),
    F.stddev("rating").alias("std_rating"),
    F.sum(F.when((F.col("rating") < 1) | (F.col("rating") > 5), 1).otherwise(0)).alias("out_of_range"),
).toPandas()

rating_summary

In [ ]:
rating_dist = (
    df.groupBy("rating")
    .count()
    .orderBy("rating")
    .toPandas()
)
rating_dist["pct"] = rating_dist["count"] / rating_dist["count"].sum()
rating_dist

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=rating_dist, x="rating", y="count", color="#F58518")
plt.xlabel("Rating")
plt.ylabel("Numero de reviews")
plt.title("Distribucion de ratings")
plt.tight_layout()

## 5. Fechas y actividad temporal

`timestamp` viene en milisegundos Unix. Convertirlo permite detectar periodos raros, cambios de volumen y posibles filtros temporales para train/test.

In [ ]:
df_dates = df.withColumn("review_date", F.to_date(F.from_unixtime(F.col("timestamp") / 1000)))

date_summary = df_dates.select(
    F.min("review_date").alias("min_date"),
    F.max("review_date").alias("max_date"),
    F.sum(F.when(F.col("timestamp").isNull(), 1).otherwise(0)).alias("missing_timestamp"),
).toPandas()

date_summary

In [ ]:
reviews_by_year = (
    df_dates.withColumn("year", F.year("review_date"))
    .groupBy("year")
    .count()
    .orderBy("year")
    .toPandas()
)

plt.figure(figsize=(10, 4))
sns.lineplot(data=reviews_by_year, x="year", y="count", marker="o", color="#54A24B")
plt.xlabel("Ano")
plt.ylabel("Numero de reviews")
plt.title("Reviews por ano")
plt.tight_layout()

reviews_by_year.tail(10)

## 6. Duplicados

Hay varias definiciones posibles de duplicado. Para recomendacion suele importar si el mismo usuario valora el mismo producto padre mas de una vez. En ese caso se puede conservar la review mas reciente o agregar ratings.

In [ ]:
duplicate_source = df.select("user_id", "parent_asin", "rating", "timestamp")

if RUN_EXPENSIVE_CHECKS:
    duplicate_scope = duplicate_source
    scope_label = "full_dataset"
    scope_rows = n_rows
    full_duplicate_rows = n_rows - duplicate_source.dropDuplicates().count()
else:
    duplicate_scope = duplicate_source.sample(False, SAMPLE_FRACTION, seed=42)
    scope_label = f"sample_{SAMPLE_FRACTION:.0%}"
    scope_rows = duplicate_scope.count()
    full_duplicate_rows = None

duplicate_keys = (
    duplicate_scope.groupBy("user_id", "parent_asin")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_key_summary = duplicate_keys.select(
    F.count("*").alias("duplicated_user_item_pairs"),
    F.sum("count").alias("rows_in_duplicated_pairs"),
    F.max("count").alias("max_reviews_same_user_item"),
).toPandas()

pd.DataFrame(
    {
        "metric": ["scope", "scope_rows", "full_duplicate_rows", *duplicate_key_summary.columns.tolist()],
        "value": [scope_label, scope_rows, full_duplicate_rows, *duplicate_key_summary.iloc[0].tolist()],
    }
)

In [ ]:
(
    duplicate_scope.groupBy("user_id", "parent_asin")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .toPandas()
)

Opciones de limpieza para pares usuario-producto repetidos:

- Mantener la review mas reciente usando `timestamp`.
- Agregar por media si se quiere representar preferencia historica.
- Eliminar duplicados exactos primero y resolver repeticiones despues.

Para ALS, una matriz usuario-item con una sola interaccion por par suele ser mas simple y estable.

## 7. Usuarios, productos y sparsity

La calidad del recomendador depende mucho de cuantas interacciones tiene cada usuario y cada producto. Usuarios o items con una sola interaccion aportan poca senal para filtrado colaborativo.

In [ ]:
entity_summary = df.select(
    F.approx_count_distinct("user_id").alias("unique_users"),
    F.approx_count_distinct("asin").alias("unique_asin"),
    F.approx_count_distinct("parent_asin").alias("unique_parent_asin"),
).toPandas()

unique_users = int(entity_summary.loc[0, "unique_users"])
unique_items = int(entity_summary.loc[0, "unique_parent_asin"])
observed_interactions = df.select("user_id", "parent_asin").dropna().dropDuplicates().count()
approx_sparsity = 1 - observed_interactions / (unique_users * unique_items)

entity_summary.assign(
    observed_user_item_pairs=observed_interactions,
    approx_sparsity=approx_sparsity,
)

In [ ]:
activity_source = df if RUN_EXPENSIVE_CHECKS else df.sample(False, SAMPLE_FRACTION, seed=42)
activity_scope = "full_dataset" if RUN_EXPENSIVE_CHECKS else f"sample_{SAMPLE_FRACTION:.0%}"

user_activity = activity_source.groupBy("user_id").count()
item_activity = activity_source.groupBy("parent_asin").count()

activity_summary = pd.DataFrame(
    {
        "scope": [activity_scope, activity_scope],
        "entity": ["user_id", "parent_asin"],
        "p01": [user_activity.approxQuantile("count", [0.01], 0.01)[0], item_activity.approxQuantile("count", [0.01], 0.01)[0]],
        "p25": [user_activity.approxQuantile("count", [0.25], 0.01)[0], item_activity.approxQuantile("count", [0.25], 0.01)[0]],
        "p50": [user_activity.approxQuantile("count", [0.50], 0.01)[0], item_activity.approxQuantile("count", [0.50], 0.01)[0]],
        "p75": [user_activity.approxQuantile("count", [0.75], 0.01)[0], item_activity.approxQuantile("count", [0.75], 0.01)[0]],
        "p95": [user_activity.approxQuantile("count", [0.95], 0.01)[0], item_activity.approxQuantile("count", [0.95], 0.01)[0]],
        "max": [user_activity.agg(F.max("count")).first()[0], item_activity.agg(F.max("count")).first()[0]],
    }
)

activity_summary

In [ ]:
user_activity_sample = user_activity.limit(100_000).toPandas()
item_activity_sample = item_activity.limit(100_000).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(user_activity_sample["count"], bins=50, log_scale=(True, True), ax=axes[0], color="#4C78A8")
axes[0].set_title("Actividad por usuario")
axes[0].set_xlabel("Reviews por usuario")
axes[0].set_ylabel("Frecuencia")

sns.histplot(item_activity_sample["count"], bins=50, log_scale=(True, True), ax=axes[1], color="#E45756")
axes[1].set_title("Actividad por producto padre")
axes[1].set_xlabel("Reviews por producto")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()

In [ ]:
top_users = user_activity.orderBy(F.desc("count")).limit(10).toPandas()
top_items = item_activity.orderBy(F.desc("count")).limit(10).toPandas()

display(top_users)
display(top_items)

## 8. Votos utiles y compra verificada

Estas columnas no son imprescindibles para ALS, pero ayudan a decidir filtros de calidad o futuras features. Por ejemplo, `verified_purchase` puede reducir ruido, aunque tambien puede sesgar el dataset.

Nota: las columnas `title`, `text` e `images` existen en el parquet crudo, pero se dejan fuera del EDA principal porque son pesadas y pueden agotar la memoria local al convertir muestras a pandas o hacer agregaciones Spark.

In [ ]:
quality_summary = df.select(
    F.mean(F.col("verified_purchase").cast("int")).alias("verified_purchase_rate"),
    F.min("helpful_vote").alias("min_helpful_vote"),
    F.max("helpful_vote").alias("max_helpful_vote"),
    F.mean("helpful_vote").alias("mean_helpful_vote"),
).toPandas()

quality_summary

In [ ]:
verified_dist = (
    df.groupBy("verified_purchase")
    .count()
    .orderBy("verified_purchase")
    .toPandas()
)

plt.figure(figsize=(5, 4))
sns.barplot(data=verified_dist, x="verified_purchase", y="count", color="#72B7B2")
plt.xlabel("Compra verificada")
plt.ylabel("Numero de reviews")
plt.title("Reviews verificadas vs no verificadas")
plt.tight_layout()

verified_dist

## 9. Comparar escenarios de limpieza

La tabla siguiente permite comparar reglas candidatas antes de cambiar el pipeline. Ajusta los umbrales y revisa cuantas filas, usuarios e items se conservan.

In [ ]:
def keep_latest_user_item_review(input_df):
    window = Window.partitionBy("user_id", "parent_asin").orderBy(F.desc("timestamp"))
    return (
        input_df.withColumn("row_number", F.row_number().over(window))
        .filter(F.col("row_number") == 1)
        .drop("row_number")
    )


def filter_min_activity(input_df, min_user_reviews=2, min_item_reviews=2):
    users = input_df.groupBy("user_id").count().filter(F.col("count") >= min_user_reviews).select("user_id")
    items = input_df.groupBy("parent_asin").count().filter(F.col("count") >= min_item_reviews).select("parent_asin")
    return input_df.join(users, "user_id", "inner").join(items, "parent_asin", "inner")


def summarize_candidate(name, input_df):
    return {
        "scope": scenario_scope,
        "scenario": name,
        "rows": input_df.count(),
        "users": input_df.select("user_id").distinct().count(),
        "items": input_df.select("parent_asin").distinct().count(),
        "avg_rating": input_df.agg(F.mean("rating")).first()[0],
    }


scenario_source = df if RUN_EXPENSIVE_CHECKS else df.sample(False, SAMPLE_FRACTION, seed=123)
scenario_scope = "full_dataset" if RUN_EXPENSIVE_CHECKS else f"sample_{SAMPLE_FRACTION:.0%}"

base_recsys = scenario_source.select("user_id", "parent_asin", F.col("rating").cast("float"), "timestamp", "verified_purchase")

candidate_required = base_recsys.dropna(subset=["user_id", "parent_asin", "rating"])
candidate_valid_rating = candidate_required.filter(F.col("rating").between(1, 5))
candidate_latest = keep_latest_user_item_review(candidate_valid_rating)
candidate_min_2 = filter_min_activity(candidate_latest, min_user_reviews=2, min_item_reviews=2)
candidate_verified = candidate_latest.filter(F.col("verified_purchase") == True)

scenarios = [
    summarize_candidate("raw", base_recsys),
    summarize_candidate("required_fields", candidate_required),
    summarize_candidate("valid_rating_1_5", candidate_valid_rating),
    summarize_candidate("latest_per_user_parent_asin", candidate_latest),
    summarize_candidate("latest_min_user_item_2", candidate_min_2),
    summarize_candidate("latest_verified_only", candidate_verified),
]

scenario_pd = pd.DataFrame(scenarios)
scenario_pd["row_retention"] = scenario_pd["rows"] / scenario_pd.loc[scenario_pd["scenario"] == "raw", "rows"].iloc[0]
scenario_pd

In [ ]:
plot_scenarios = scenario_pd.sort_values("row_retention", ascending=True)

plt.figure(figsize=(9, 4))
sns.barplot(data=plot_scenarios, x="row_retention", y="scenario", color="#B279A2")
plt.gca().xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
plt.xlabel("Retencion de filas")
plt.ylabel("")
plt.title("Impacto de reglas candidatas de limpieza")
plt.tight_layout()

## 10. Dataset candidato para modelado

Esta celda crea una version candidata sin guardarla. Si las metricas anteriores son razonables, las reglas deberian moverse al pipeline de Kedro en `src/amazon_recsys/pipelines/data_processing/nodes.py`.

In [ ]:
clean_candidate = (
    candidate_latest
    .select("user_id", "parent_asin", "rating")
    .dropna(subset=["user_id", "parent_asin", "rating"])
)

clean_candidate.printSchema()
clean_candidate.limit(10).toPandas()

## 11. Decisiones de limpieza a cerrar

Checklist recomendado antes de editar el pipeline:

- Confirmar item id: usar `parent_asin` para agrupar variantes o `asin` para productos concretos.
- Confirmar duplicados: conservar review mas reciente por `user_id` + `parent_asin`, o agregar rating medio.
- Confirmar filtro de rating: conservar solo valores entre 1 y 5.
- Confirmar minimo de actividad: evaluar umbrales como 2, 3 o 5 interacciones por usuario/item segun retencion.
- Confirmar si `verified_purchase == True` sera filtro duro o solo feature de calidad.
- Confirmar estrategia temporal: usar fechas para split train/test si se evaluara recomendacion con holdout temporal.

Decision base sugerida para empezar: `user_id`, `parent_asin`, `rating`; eliminar nulos; filtrar ratings 1-5; resolver duplicados manteniendo la review mas reciente. Dejar `verified_purchase` y minimos de actividad como experimentos porque pueden reducir mucho la cobertura.